# 🏦 Bank Customer Churn Prediction — Model Training
Trains **Random Forest**, **XGBoost**, and **LightGBM** models on the Bank Churn dataset and saves them to `model.pkl` for the Streamlit application.

In [1]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from sklearn.metrics import accuracy_score, classification_report

## 1. Load and Prepare Data

In [2]:
df = pd.read_csv("Bank_Churn.csv")

# Drop unnecessary identifier columns
df = df.drop(columns=["CustomerId", "Surname"], errors="ignore")
df = df.dropna()

X = df.drop(columns="Exited")
y = df["Exited"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Dataset shape: {df.shape}")
print(f"Train samples: {len(X_train)} | Test samples: {len(X_test)}")
df.head()

Dataset shape: (10000, 11)
Train samples: 8000 | Test samples: 2000


,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


## 2. Preprocessing Pipeline
StandardScaler for numerical features and OneHotEncoder for categorical features.

In [3]:
numeric_features = X.select_dtypes(include="number").columns.tolist()
categorical_features = X.select_dtypes(exclude="number").columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

## 3. Train Random Forest

In [4]:
rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=42))
])

print("Training Random Forest...")
rf_pipeline.fit(X_train, y_train)

rf_pred = rf_pipeline.predict(X_test)
print(f"Random Forest Accuracy: {accuracy_score(y_test, rf_pred):.4f}")
print(classification_report(y_test, rf_pred, target_names=["Stay", "Churn"]))

Training Random Forest...


Random Forest Accuracy: 0.8600
              precision    recall  f1-score   support

        Stay       0.87      0.97      0.92      1593
       Churn       0.77      0.44      0.56       407

    accuracy                           0.86      2000
   macro avg       0.82      0.71      0.74      2000
weighted avg       0.85      0.86      0.84      2000



## 4. Train XGBoost

In [5]:
xgb_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", XGBClassifier(random_state=42, eval_metric="logloss"))
])

print("Training XGBoost...")
xgb_pipeline.fit(X_train, y_train)

xgb_pred = xgb_pipeline.predict(X_test)
print(f"XGBoost Accuracy: {accuracy_score(y_test, xgb_pred):.4f}")
print(classification_report(y_test, xgb_pred, target_names=["Stay", "Churn"]))

Training XGBoost...


XGBoost Accuracy: 0.8450
              precision    recall  f1-score   support

        Stay       0.88      0.94      0.91      1593
       Churn       0.67      0.48      0.56       407

    accuracy                           0.84      2000
   macro avg       0.77      0.71      0.73      2000
weighted avg       0.83      0.84      0.84      2000



## 5. Train LightGBM

In [6]:
lgb_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LGBMClassifier(random_state=42, verbose=-1))
])

print("Training LightGBM...")
lgb_pipeline.fit(X_train, y_train)

lgb_pred = lgb_pipeline.predict(X_test)
print(f"LightGBM Accuracy: {accuracy_score(y_test, lgb_pred):.4f}")
print(classification_report(y_test, lgb_pred, target_names=["Stay", "Churn"]))

Training LightGBM...


LightGBM Accuracy: 0.8595
              precision    recall  f1-score   support

        Stay       0.88      0.96      0.92      1593
       Churn       0.74      0.48      0.58       407

    accuracy                           0.86      2000
   macro avg       0.81      0.72      0.75      2000
weighted avg       0.85      0.86      0.85      2000



## 6. Save Models for Streamlit App

In [7]:
models = {
    "Random Forest": rf_pipeline,
    "XGBoost": xgb_pipeline,
    "LightGBM": lgb_pipeline,
}

joblib.dump(models, "model.pkl")
print("✔ Successfully saved all 3 models to model.pkl!")

✔ Successfully saved all 3 models to model.pkl!
